https://datapilots.tistory.com/59 를 참고해서 만듦.

| 데이터명 | 사용 목적 | merge 기준 컬럼 |
| :--- | :--- | :--- |
| PORT MIS 입출항 현황 | 입출항 데이터 사용 > 서비스시간 산출 | 호출부호, 선박명 , 입항횟수_횟수, 입항횟수_연도 |
| PORT MIS 시설사용허가현황 | 접안대기, 출항대기 데이터 파악 및 대기시간 산출 | |


-> 기존 프로젝트와 차별점
입출항 현황 데이터에 '차항지', '전출항지', 항해구분', '선박용도' 컬럼을 추가함

In [10]:
!pip install openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 불러오기 & 시설사용허가현황 컬럼 정리

In [11]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

directory = './data'
dfs_facility = []

for i in range(12):
    file_path = f'{directory}/download({i}).xlsx'
    try:
        df = pd.read_excel(file_path, header=None)

        # 4행(Index 3)을 헤더로 지정
        headers = df.iloc[3].values
        df_data = df.iloc[4:].copy()
        df_data.columns = [str(h).strip() for h in headers]

        # ---------------------------------------------------------
        # 중복 컬럼명 순서대로 1:1 매핑 (FROM 1개, TO 1개만 할당)
        # ---------------------------------------------------------
        cols = list(df_data.columns)

        count_call = 0
        count_period = 0

        for idx, col in enumerate(cols):
            # 입항횟수 (년도, 횟수 구분)
            if '입항횟수' in col:
                if count_call == 0:
                    cols[idx] = '입항횟수_년도'
                    count_call += 1
                elif count_call >= 1:
                    cols[idx] = '입항횟수_횟수'

            # 기간/일시 컬럼 처리 (첫 번째 세트만 FROM / TO로 지정)
            elif any(keyword in col for keyword in ['지정', '기간', '일시']) and col not in ['신청일시', '수리일시', '입항일시', '출항일시']:
                if '시설' not in col and '코드' not in col: # 지정시설 제거
                    if count_period == 0:
                        cols[idx] = '지정일시_FROM'
                        count_period += 1
                    elif count_period == 1:
                        cols[idx] = '지정일시_TO'
                        count_period += 1
                    else:
                        cols[idx] = f'기타기간_{count_period}' # 3번째부터는 다른 이름으로 밀어냄
                        count_period += 1

        df_data.columns = cols

        # 중복 컬럼 제거 (동일한 컬럼명이 생기지 않도록 방지)
        df_data = df_data.loc[:, ~df_data.columns.duplicated()]

        dfs_facility.append(df_data)

    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {file_path}")

# 데이터 통합
facility_df = pd.concat(dfs_facility, ignore_index=True)

# ---------------------------------------------------------
# 필요한 핵심 컬럼만 추출
# ---------------------------------------------------------
target_cols = [
    '호출부호', '선박명', '입항횟수_년도', '입항횟수_횟수', '신고톤수',
    '지정시설', '지정일시_FROM', '지정일시_TO', '사용목적명', '신청일시', '허가유무'
]

# 실제 존재하는 컬럼만 선별
existing_cols = [c for c in target_cols if c in facility_df.columns]
facility_df = facility_df[existing_cols]

# 미허가 데이터 제거
if '허가유무' in facility_df.columns:
    facility_df = facility_df[facility_df['허가유무'] != '미허가']

print("최종 정리된 컬럼 목록:", facility_df.columns.tolist())
facility_df.head()

최종 정리된 컬럼 목록: ['호출부호', '선박명', '입항횟수_년도', '입항횟수_횟수', '신고톤수', '지정시설', '지정일시_FROM', '지정일시_TO', '사용목적명', '신청일시', '허가유무']


,호출부호,선박명,입항횟수_년도,입항횟수_횟수,신고톤수,지정시설,지정일시_FROM,지정일시_TO,사용목적명,신청일시,허가유무
0,215802,장군2호,2024,001,39,*JG,2024-01-30 17:40,2024-01-31 07:30,출항대기,2024-01-31 22:34,허가완료
1,UBCZ2,MERIDA,2024,001,5.275,WAN,2024-02-01 10:00,2024-02-01 19:00,급유,2024-01-31 22:29,허가완료
2,3WON,SEA DREAM,2024,001,8.216,WAN,2024-02-01 05:00,2024-02-01 15:00,급유,2024-01-31 22:28,허가완료
3,D7XB,PANCON BRIDGE,2024,002,18.040,MBS,2024-02-01 04:00,2024-02-02 13:00,접안대기,2024-01-31 22:28,허가완료
4,DSRL4,KMTC SHANGHAI,2024,002,20.815,WAN,2024-02-01 00:01,2024-02-02 10:00,접안대기,2024-01-31 22:26,허가완료


-------------------------------------


# 입출항 현황 데이터 정리

In [9]:
# ---------------------------------------------------------
# 2. 입출항 데이터 불러오기 및 통합
# ---------------------------------------------------------
ship_files = [f'{directory}/입출항(1).xlsx', f'{directory}/입출항(2).xlsx']
dfs_ship = []

for file_path in ship_files:
    try:
        df = pd.read_excel(file_path)
        # 각 파일마다 상단 11행(제목 영역) 삭제
        df = df.iloc[11:].copy()
        dfs_ship.append(df)
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {file_path}")

# 입출항 2개 파일 병합
ship_df = pd.concat(dfs_ship, ignore_index=True)

# 컬럼명 지정 (선박입출항현황 서식 기준)
ship_df.columns = [
    '항명', '호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수', '구분', '외내', '입출',
    '총톤수', '입항일시', '출항일시', 'CIQ수속일자', '수리일시', '항해구분', 'MRN 번호',
    '계선장소_코드', '계선장소_숫자', '계선장소명', '차항지', '전출항지', '선박용도',
    '외항:한국인선원수/내항:해기사선원수', '외항:외국인선원수/내항:보통선원수',
    '승무원승선수', '승무원하선수', '승객승선수', '승객하선수', '승객', '예선', '도선',
    '부선호출부호1', '부선호출부호2'
]

# 불필요한 컬럼 제거 및 '출항' 기준 필터링
columns_to_drop_ship = [
    '외내', 'CIQ수속일자', 'MRN 번호',
    '외항:한국인선원수/내항:해기사선원수', '외항:외국인선원수/내항:보통선원수',
    '승무원승선수', '승무원하선수', '승객승선수', '승객하선수', '승객', '예선', '도선',
    '부선호출부호1', '부선호출부호2'
]
ship_df.drop(columns=[c for c in columns_to_drop_ship if c in ship_df.columns], inplace=True)
ship_df = ship_df[ship_df['입출'] == '출항']



ship_df.head()

,항명,호출부호,선박명,입항횟수_연도,입항횟수_횟수,구분,입출,총톤수,입항일시,출항일시,수리일시,항해구분,계선장소_코드,계선장소_숫자,계선장소명,차항지,전출항지,선박용도
0,부산,796256,백경3,2004,001,최종,출항,22,2004-01-01 00:00,2024-10-23 08:00,2024-11-08 16:10,기타,*YQ,01,기타 남항 사설조선소,기타항,기타항,석유제품 운반선
1,부산,058251,대은호,2016,002,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,NaN,기타,MW5,01,5물량장 1선석,기타항,고현,급유선
2,부산,915291,17금영호,2019,001,최종,출항,22,2019-05-30 09:00,2024-08-29 15:00,2024-09-02 15:59,폐선예정,MQF,02,감천 선기조합조선소 안벽-02,기타항,삼천포,급유선
3,부산,JVNF8,SUNRISE,2019,001,최종,출항,19,2019-11-21 18:00,2024-05-10 11:30,2024-07-11 10:15,기타,WAS,01,수영박지(집단박지),속초,HAKATA/FUKUOKA,수상레저기구
4,부산,030713,청운호,2021,001,최초,출항,121,2021-01-01 00:00,2024-09-30 00:00,NaN,기타,MW5,01,5물량장 1선석,기타항,NaN,석유제품 운반선


# 2개 데이터 합치기

In [14]:
import pandas as pd

# ---------------------------------------------------------
# 1. 컬럼명 통일 및 중복 컬럼 정리 (핵심!)
# ---------------------------------------------------------
# facility_df와 ship_df의 입항연도 컬럼명을 '입항횟수_연도'로 통일
if '입항횟수_년도' in facility_df.columns:
    facility_df.rename(columns={'입항횟수_년도': '입항횟수_연도'}, inplace=True)

if '입항횟수_년도' in ship_df.columns:
    ship_df.rename(columns={'입항횟수_년도': '입항횟수_연도'}, inplace=True)

# 병합 중복 등으로 인한 중복 컬럼 제거 (첫번째로 나온 컬럼만 남김)
facility_df = facility_df.loc[:, ~facility_df.columns.duplicated()].copy()
ship_df = ship_df.loc[:, ~ship_df.columns.duplicated()].copy()

# ---------------------------------------------------------
# 2. Key 컬럼 타입 일치화 (Numeric 변환)
# ---------------------------------------------------------
for df in [facility_df, ship_df]:
    df['입항횟수_횟수'] = pd.to_numeric(df['입항횟수_횟수'], errors='coerce')
    df['입항횟수_연도'] = pd.to_numeric(df['입항횟수_연도'], errors='coerce')

# ---------------------------------------------------------
# 3. 데이터 병합 (Left Merge)
# ---------------------------------------------------------
merged_df = pd.merge(
    ship_df, facility_df,
    on=['호출부호', '선박명', '입항횟수_연도', '입항횟수_횟수'],
    how='left'
)

# 중복 데이터 제거
merged_df = merged_df.drop_duplicates()

# 항만권역
import pandas as pd

# 1. 원본 복사 및 최종 계선코드 생성
merged_df_1 = merged_df.copy()
merged_df_1['최종_계선코드'] = merged_df_1['지정시설'].fillna(merged_df_1['계선장소_코드'])

# 2. 권역 구분 함수
def get_port_zone(row):
    code = str(row['최종_계선코드']).strip().upper() if pd.notna(row['최종_계선코드']) else ''
    name = str(row['계선장소명']).strip() if pd.notna(row['계선장소명']) else ''

    combined_text = f"{code} {name}".upper()

    if not combined_text.strip():
        return '기타/미지정'

    if any(k in combined_text for k in ['PNIT', 'PNC', 'HPNT', 'BNCT', 'HJNC', 'DPS', '신항']):
        return '신항'
    elif any(k in combined_text for k in ['MW5', '*YQ', 'WAS', '신선대', '감만', '자성대', '우암', '북항', '영도', '초량', '부산역']):
        return '북항'
    elif any(k in combined_text for k in ['MKD', 'MK1', 'MK2', 'MK3', '감천', '다대포']):
        return '감천항'
    else:
        return '기타/일반계류지'

# 3. 새 컬럼 '항만권역' 생성
merged_df_1['항만권역'] = merged_df_1.apply(get_port_zone, axis=1)

# 4. 기존 컬럼만 삭제 (최종_계선코드, 항만권역은 보존)
cols_to_drop = ['지정시설', '계선장소_코드', '계선장소_숫자', '계선장소명']
merged_df_1 = merged_df_1.drop(columns=[c for c in cols_to_drop if c in merged_df_1.columns])



merged_df_1.head()

,항명,호출부호,선박명,입항횟수_연도,입항횟수_횟수,구분,입출,총톤수,입항일시,출항일시,...,전출항지,선박용도,신고톤수,지정일시_FROM,지정일시_TO,사용목적명,신청일시,허가유무,최종_계선코드,항만권역
0,부산,796256,백경3,2004,1,최종,출항,22,2004-01-01 00:00,2024-10-23 08:00,...,기타항,석유제품 운반선,22,2004-01-01 00:00,2013-12-31 00:00,기타,2024-12-03 11:17,허가완료,MW5,북항
1,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2024-07-16 00:00,2024-07-31 23:59,계선,2024-07-15 11:25,허가완료,MW5,북항
2,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2024-07-01 00:00,2024-07-15 23:59,계선,2024-07-15 11:25,허가완료,MW5,북항
3,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2017-05-11 08:55,2018-07-13 09:40,계선,2024-11-26 10:51,허가완료,MW5,북항
4,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2016-11-15 11:35,2016-11-16 10:40,기타,2024-11-26 10:44,허가완료,MW5,북항


# 파일로 저장
매번 CSV 파일들 불러와서 컬럼 정리하고 MERGE 하기 귀찮음.

In [15]:
import os

# processed_data 폴더가 없으면 자동 생성
os.makedirs('./processed_data', exist_ok=True)

# CSV 파일로 저장 (인덱스 제외, 한글 깨짐 방지 utf-8-sig 적용)
merged_df_1.to_csv('./processed_data/merged_df.csv', index=False, encoding='utf-8-sig')

print(f"merged_df_1.csv 저장 완료! 총 {len(merged_df_1)}건의 데이터가 저장되었습니다.")

merged_df_1.csv 저장 완료! 총 85483건의 데이터가 저장되었습니다.


-----


In [16]:
# test용 파일
merged_df_1.head(50).to_csv('./processed_data/merged_df(head).csv', index=False, encoding='utf-8-sig')

In [17]:
merged_df_1.head(50)

,항명,호출부호,선박명,입항횟수_연도,입항횟수_횟수,구분,입출,총톤수,입항일시,출항일시,...,전출항지,선박용도,신고톤수,지정일시_FROM,지정일시_TO,사용목적명,신청일시,허가유무,최종_계선코드,항만권역
0,부산,796256,백경3,2004,1,최종,출항,22,2004-01-01 00:00,2024-10-23 08:00,...,기타항,석유제품 운반선,22,2004-01-01 00:00,2013-12-31 00:00,기타,2024-12-03 11:17,허가완료,MW5,북항
1,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2024-07-16 00:00,2024-07-31 23:59,계선,2024-07-15 11:25,허가완료,MW5,북항
2,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2024-07-01 00:00,2024-07-15 23:59,계선,2024-07-15 11:25,허가완료,MW5,북항
3,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2017-05-11 08:55,2018-07-13 09:40,계선,2024-11-26 10:51,허가완료,MW5,북항
4,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2016-11-15 11:35,2016-11-16 10:40,기타,2024-11-26 10:44,허가완료,MW5,북항
5,부산,058251,대은호,2016,2,최종,출항,124,2016-08-16 16:15,2024-12-31 23:59,...,고현,급유선,124,2016-11-04 10:00,2016-11-15 08:30,기타,2024-11-26 10:44,허가완료,MW5,북항
6,부산,915291,17금영호,2019,1,최종,출항,22,2019-05-30 09:00,2024-08-29 15:00,...,삼천포,급유선,22,2024-07-24 08:00,2024-07-28 18:00,기타,2024-07-23 13:26,허가완료,MW5,북항
7,부산,JVNF8,SUNRISE,2019,1,최종,출항,19,2019-11-21 18:00,2024-05-10 11:30,...,HAKATA/FUKUOKA,수상레저기구,19,2019-11-21 16:00,2019-11-21 17:00,수리,2024-07-11 10:12,허가완료,WAS,북항
8,부산,030713,청운호,2021,1,최초,출항,121,2021-01-01 00:00,2024-09-30 00:00,...,NaN,석유제품 운반선,NaN,NaN,NaN,NaN,NaN,NaN,MW5,북항
9,부산,DSDN4,성진1호,2021,1,최종,출항,155,2021-08-18 12:10,2024-12-14 10:55,...,인천,견인용예선,155,2021-08-18 12:10,2024-12-11 18:00,계선,2024-12-11 17:34,허가완료,*NB,기타/일반계류지
